# 02 Data quality

**In:** `orders_cl.csv`, `orderlines_cl.csv` and `products_cl.csv` (folder `data/clean-data`, made by `01_data-cleaning.ipynb`).
**Out:** `orders_qu.csv` and `orderlines_qu.csv` (folder `data/clean-data`).

Notebook 01 fixed problems inside each table. This notebook checks that the data makes **business sense across the tables**, and keeps only the orders we can trust:
1. Completed orders only, with a total above €0 and no product priced at €0 or less
2. Orders that appear in both `orders` and `orderlines`
3. Orders whose products are all in the `products` table
4. Orders where the amount paid matches the products bought (apart from a normal shipping fee)

`products_cl.csv` is not changed here. `03_product-categories.ipynb` adds the categories to it.

## Setup

In [5]:
import pandas as pd

In [6]:
# show numbers with 2 decimals, and allow long tables
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_rows", 1000)

In [7]:
orders_cl = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/clean-data/orders_cl.csv")
orderlines_cl = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/clean-data/orderlines_cl.csv")
products_cl = pd.read_csv("https://raw.githubusercontent.com/AliHaydar2023/eniac-discount-analysis/main/data/clean-data/products_cl.csv")

In [8]:
# size of each clean table (rows, columns)
orders_cl.shape, orderlines_cl.shape, products_cl.shape

((226904, 4), (216250, 7), (9992, 6))

## Step 1: Prepare the tables

In [9]:
orders_s1 = orders_cl.copy()
orderlines_s1 = orderlines_cl.copy()
products_s1 = products_cl.copy()

In [10]:
# a CSV file stores dates as text, so turn them back into dates
orders_s1["created_date"] = pd.to_datetime(orders_s1["created_date"])
orderlines_s1["date"] = pd.to_datetime(orderlines_s1["date"])
orders_s1["created_date"].dtype, orderlines_s1["date"].dtype

(dtype('<M8[ns]'), dtype('<M8[ns]'))

In [11]:
# every value in product_id
orderlines_s1["product_id"].value_counts()

,count
product_id,
0,216250


In [12]:
# product_id is always 0, so it tells us nothing: remove it (sku identifies the product)
orderlines_s1 = orderlines_s1.drop(columns=["product_id"])
orderlines_s1.columns

Index(['id', 'id_order', 'product_quantity', 'sku', 'unit_price', 'date'], dtype='object')

**What we see:**
- The dates are dates again.
- `product_id` is 0 on all 216,250 order lines, so it is removed. Products are identified by `sku`.

## Step 2: Remove unwanted orders

In [13]:
orders_s2 = orders_s1.copy()
orderlines_s2 = orderlines_s1.copy()
products_s2 = products_s1.copy()

### Orders: Completed only, total above €0

In [14]:
# number of orders in each state
orders_s2["state"].value_counts()

,count
state,
Shopping Basket,117809
Completed,46605
Place Order,40883
Pending,14374
Cancelled,7233


In [15]:
# keep only Completed orders: the others were never paid for (baskets, cancelled, pending)
orders_s2 = orders_s2.loc[orders_s2["state"] == "Completed", :]
orders_s2.shape

(46605, 4)

In [16]:
# Completed orders with a total of exactly €0, and with a negative total
(orders_s2["total_paid"] == 0).sum(), (orders_s2["total_paid"] < 0).sum()

(np.int64(16), np.int64(0))

In [17]:
# remove the Completed orders with a total of €0
orders_s2 = orders_s2.loc[orders_s2["total_paid"] != 0, :]
orders_s2.shape

(46589, 4)

**What we see:**
- Only 46,605 of 226,904 orders (21%) are Completed. The largest group is Shopping Basket (117,809): baskets that were never bought.
- 16 Completed orders have a total of €0, which makes no sense for a sale. They are removed: 46,589 orders are left.
- No order has a negative total.

### Orderlines: quantities and prices

In [18]:
# order lines with a quantity of 0 or less
(orderlines_s2["product_quantity"] <= 0).sum()

np.int64(0)

In [19]:
# order lines priced at exactly €0, and below €0
(orderlines_s2["unit_price"] == 0).sum(), (orderlines_s2["unit_price"] < 0).sum()

(np.int64(597), np.int64(1))

In [20]:
# the orders that contain a line priced at €0 or less: number of lines, number of orders
orders_with_bad_price = orderlines_s2.loc[orderlines_s2["unit_price"] <= 0, "id_order"]
len(orders_with_bad_price), orders_with_bad_price.nunique()

(598, 258)

In [21]:
# remove these orders completely
orderlines_s2 = orderlines_s2.loc[~orderlines_s2["id_order"].isin(orders_with_bad_price), :]
orderlines_s2.shape

(215358, 6)

**What we see:**
- All quantities are 1 or more.
- 597 order lines have a price of €0 and 1 has a negative price. A product can't be sold for €0 or less, so the 258 orders containing these 598 lines are removed.
- 215,358 order lines are left.

### Keep only orders that are in both tables

In [22]:
# number of different orders in each table
orders_s2["order_id"].nunique(), orderlines_s2["id_order"].nunique()

(46589, 169955)

In [23]:
# an inner merge keeps only the orders found in both tables
orders_orderlines = orders_s2.merge(orderlines_s2, how="inner", left_on="order_id", right_on="id_order")
order_ids = orders_orderlines["order_id"].unique()

# keep those orders in both tables
orders_s2 = orders_s2.loc[orders_s2["order_id"].isin(order_ids), :]
orderlines_s2 = orderlines_s2.loc[orderlines_s2["id_order"].isin(order_ids), :]

orders_s2["order_id"].nunique(), orderlines_s2["id_order"].nunique()

(43053, 43053)

In [24]:
# size of each table after step 2
orders_s2.shape, orderlines_s2.shape

((43053, 4), (56687, 6))

**What we see:**
- Before this check, orderlines held lines from 169,955 orders, because it still had the lines of baskets, cancelled and pending orders.
- 43,053 orders are in both tables, now with 56,687 order lines.
- 3,536 Completed orders had no order lines left, so they are removed too. The next cell shows why.

In [25]:
# why these 3,536 orders have no lines: compare with the raw orderlines file
orders_without_lines = orders_s1.loc[(orders_s1["state"] == "Completed") & (orders_s1["total_paid"] != 0), "order_id"]
orders_without_lines = orders_without_lines[~orders_without_lines.isin(orderlines_s2["id_order"])]
len(orders_without_lines), orders_without_lines.isin(orderlines_cl["id_order"]).sum()

(3536, np.int64(9))

**What we see:** of the 3,536 orders, only 9 lost their lines in this step (a price of €0 or less). The other 3,527 already had no lines in `orderlines_cl`: notebook 01 removed every order with a two-dot price, which was 34,642 orders.

## Step 3: Remove orders with unknown products

In [26]:
orders_s3 = orders_s2.copy()
orderlines_s3 = orderlines_s2.copy()
products_s3 = products_s2.copy()

In [27]:
# a left merge keeps every order line; name stays empty (NaN) when the sku is not in products
orderlines_products = orderlines_s3.merge(products_s3, how="left", on="sku")[["id_order", "sku", "name"]]
orderlines_products["name"].isna().value_counts()

,count
name,
False,55304
True,1383


In [28]:
# the orders with at least one unknown product
orders_with_unknown = orderlines_products.loc[orderlines_products["name"].isna(), "id_order"].unique()
len(orders_with_unknown)

1354

In [29]:
# remove these orders from both tables
orders_s3 = orders_s3.loc[~orders_s3["order_id"].isin(orders_with_unknown), :]
orderlines_s3 = orderlines_s3.loc[~orderlines_s3["id_order"].isin(orders_with_unknown), :]

orders_s3["order_id"].nunique(), orderlines_s3["id_order"].nunique()

(41699, 41699)

In [30]:
# size of each table after step 3
orders_s3.shape, orderlines_s3.shape

((41699, 4), (54322, 6))

**What we see:**
- 1,383 order lines have a product (`sku`) that is not in the products table, spread over 1,354 orders.
- We can't tell what was sold or at what list price, so the whole order is removed (not just the line): 41,699 orders and 54,322 order lines are left.

## Step 4: Check what customers paid

In [31]:
orders_s4 = orders_s3.copy()
orderlines_s4 = orderlines_s3.copy()
products_s4 = products_s3.copy()

In [32]:
# what each order line cost: price × quantity
orderlines_s4["unit_price_total"] = orderlines_s4["unit_price"] * orderlines_s4["product_quantity"]
orderlines_s4[["unit_price", "product_quantity", "unit_price_total"]].head()

,unit_price,product_quantity,unit_price_total
6,47.49,1,47.49
7,18.99,1,18.99
8,72.19,1,72.19
15,60.90,1,60.90
21,142.49,1,142.49


In [33]:
# add up the order lines of each order
orderlines_total_price = orderlines_s4.groupby("id_order").agg(total_price=("unit_price_total", "sum"))
orderlines_total_price.head()

,total_price
id_order,
241423,129.16
242832,10.77
243330,77.99
245275,149.00
245595,105.98


In [34]:
# what the customer paid for each order
orders_total_paid = orders_s4[["order_id", "total_paid"]].set_index("order_id")
orders_total_paid.head()

,total_paid
order_id,
241423,136.15
242832,15.76
243330,84.98
245275,149.00
245595,112.97


In [35]:
# difference = paid - cost of the products (a positive difference is usually shipping)
diff_df = orders_total_paid["total_paid"].sub(orderlines_total_price["total_price"]).reset_index(name="difference")
diff_df["difference"].describe()

,difference
count,41699.00
mean,4.48
std,26.05
min,-165.00
25%,0.00
50%,4.99
75%,6.99
max,3984.99


In [36]:
# the most common differences
diff_df["difference"].round(2).value_counts().head(10)

,count
difference,
0.00,11043
4.99,10959
6.99,9921
3.99,6792
19.99,539
9.99,441
0.01,330
7.00,261
-0.01,221


**What we see:**
- For half of all orders, the customer paid €4.99 or less on top of the products. The most common differences (€0, €4.99, €6.99 …) look like shipping fees.
- But the extremes don't look like shipping: one customer paid €3,984.99 more than the products cost, another €165 less. We can't explain these from the data.

### Remove the unexplained differences (IQR method)

The IQR (interquartile range) is the spread of the middle half of the differences: from the 25% mark (Q1) to the 75% mark (Q3).
A difference counts as an outlier if it lies more than 1.5 × IQR below Q1 or above Q3. These limits are the "whiskers" of a boxplot.

In [37]:
# the 25% and 75% marks, the IQR, and the whiskers
q1 = diff_df["difference"].quantile(0.25)
q3 = diff_df["difference"].quantile(0.75)
iqr = q3 - q1
lower_whisker = q1 - 1.5 * iqr
upper_whisker = q3 + 1.5 * iqr
q1, q3, iqr, lower_whisker, upper_whisker

(np.float64(0.0),
 np.float64(6.989999999999981),
 np.float64(6.989999999999981),
 np.float64(-10.484999999999971),
 np.float64(17.47499999999995))

In [38]:
# number of orders outside the whiskers
outlier_mask = (diff_df["difference"] < lower_whisker) | (diff_df["difference"] > upper_whisker)
outlier_mask.sum()

np.int64(714)

In [39]:
# the most common differences among the outliers
diff_df.loc[outlier_mask, "difference"].round(2).value_counts().head(5)

,count
difference,
19.99,539
19.98,43
20.00,37
-20.01,11
-21.01,7


In [40]:
# keep only the orders inside the whiskers, in both tables
orders_to_keep = diff_df.loc[~outlier_mask, "order_id"]
orders_s4 = orders_s4.loc[orders_s4["order_id"].isin(orders_to_keep), :]
orderlines_s4 = orderlines_s4.loc[orderlines_s4["id_order"].isin(orders_to_keep), :]

orders_s4["order_id"].nunique(), orderlines_s4["id_order"].nunique()

(40985, 40985)

In [41]:
# the differences that are left
diff_df.loc[~outlier_mask, "difference"].describe()

,difference
count,40985.00
mean,3.94
std,2.83
min,-5.00
25%,0.00
50%,4.99
75%,6.99
max,16.97


**What we see:**
- The whiskers are −€10.48 and +€17.47. 714 orders (1.7%) fall outside them and are removed: 40,985 orders are left.
- The differences left run from −€5.00 to +€16.97, with the same typical values as before (median €4.99).
- **Caveat:** 539 of the 714 removed orders have a difference of exactly €19.99. That is probably a real (express) shipping fee, not an error. We use the IQR method because it is the team's agreed method, so everyone works with the same data, but this removes some genuine orders.

## Step 5: Final check and export

In [42]:
orders_qu = orders_s4.copy()
orderlines_qu = orderlines_s4.copy()

In [43]:
# final size of each table
orders_qu.shape, orderlines_qu.shape

((40985, 4), (53231, 7))

In [44]:
# checks: only Completed orders, the same orders in both tables, every sku known, no price of €0 or less
(
    orders_qu["state"].unique(),
    set(orders_qu["order_id"]) == set(orderlines_qu["id_order"]),
    orderlines_qu["sku"].isin(products_cl["sku"]).all(),
    (orderlines_qu["unit_price"] <= 0).sum(),
)

(array(['Completed'], dtype=object), True, np.True_, np.int64(0))

**What we see:**

| Step | Orders | Order lines |
|---|---|---|
| Start (`_cl` files) | 226,904 | 216,250 |
| 2. Completed only | 46,605 | |
| 2. Remove €0 totals | 46,589 | |
| 2. Remove lines priced €0 or less | | 215,358 |
| 2. In both tables | 43,053 | 56,687 |
| 3. Remove unknown products | 41,699 | 54,322 |
| 4. Remove unexplained differences (IQR) | **40,985** | **53,231** |

- All checks pass: only Completed orders, the same 40,985 orders in both tables, every product known, no price of €0 or less.
- `orderlines_qu` has the new column `unit_price_total` (price × quantity).

In [45]:
# save the quality-controlled tables as CSV files
orders_qu.to_csv("orders_qu.csv", index=False)
orderlines_qu.to_csv("orderlines_qu.csv", index=False)

In [46]:
# download them from Colab (then upload to data/clean-data)
from google.colab import files
files.download("orders_qu.csv")
files.download("orderlines_qu.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>